In [1]:
from dotenv import load_dotenv
load_dotenv()
import yfinance as yf

from langchain_groq import ChatGroq
from langgraph.graph import StateGraph, START, END, MessagesState
from langgraph.checkpoint.memory import MemorySaver
from typing import TypedDict, List, Literal
from langgraph.prebuilt import tools_condition, ToolNode
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, SystemMessage
import asyncio

In [2]:
#model
llm  = ChatGroq(model="openai/gpt-oss-20b")

In [3]:
#creating a tool
@tool
def get_stock_price(ticker: str) -> float | None:
    """
    Fetches the latest closing/current market price for a given ticker symbol.
    Example tickers: 'AAPL', 'MSFT', 'GOOGL', 'RELIANCE.NS' (for NSE India).
    """
    try:
        stock = yf.Ticker(ticker)
        # Fast retrieval via recent minute/day history
        data = stock.history(period="1d")
        if not data.empty:
            return round(data['Close'].iloc[-1], 2)
        
        # Fallback to ticker info
        info = stock.info
        price = info.get("regularMarketPrice") or info.get("currentPrice")
        return round(price, 2) if price else None

    except Exception as e:
        print(f"Error fetching data for {ticker}: {e}")
        return None

In [4]:
tools = [get_stock_price]

In [5]:
#bind with llm
llm_with_tools = llm.bind_tools(tools)

In [15]:
# function which will build the graph
def build_graph():
    
    #creating chatnode
    async def chat_node(state:MessagesState):
        response = await llm_with_tools.ainvoke(state["messages"])
        return {"messages": [response]}

    #adding node
    graph = StateGraph(MessagesState)
    graph.add_node("chat", chat_node)
    graph.add_node('tools', ToolNode(tools))
    
    #adding egdes
    graph.add_edge(START, 'chat')
    graph.add_conditional_edges('chat', tools_condition)
    graph.add_edge('tools','chat')
    graph.add_edge('chat', END)

    graph = graph.compile()
    return graph

In [16]:
async def main():
    #build graph
    chatbot = build_graph()

    #invoke chatbot
    res = await chatbot.ainvoke({'messages':HumanMessage('what is the current stock price of NVIDIA ?')})

    print(res['messages'][-1].content)

In [17]:
await main()

The current price of NVIDIA (NVDA) is **$225.73**.
